# 05 — Finalize BIDS Directory & Unified Dataset Manifest

Before running `fMRIPrep`, this notebook performs final BIDS validation/cleanup and merges our dataset metadata into a single unified manifest:

1. **Remove duplicate non-BIDS `_T1wa` files** in `data/raw_bids/sub-006S4485/ses-20180925/anat/` and verify zero non-standard files exist across `data/raw_bids`.
2. **Remove the 11 failed fMRI session folders (`anat`-only sessions)** from `data/raw_bids` so that `fMRIPrep` only processes the `3,267` complete `(fMRI + T1w)` sessions.
3. **Generate `data/raw_bids/participants.tsv` and `participants.json`** for full BIDS specification compliance.
4. **Combine all columns from `clean_dedup_fmri_to_t1_master.csv` and `bids_conversion_manifest.csv`** into a single comprehensive CSV: `results/dataset_creation/final_bids_dataset_manifest.csv` (and sync `Diagnosis` into `bids_conversion_manifest.csv`).

In [1]:
import json
import shutil
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'raw_bids').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

BIDS_ROOT = PROJECT_ROOT / 'data' / 'raw_bids'
MASTER_CSV = PROJECT_ROOT / 'results' / 'dataset_creation' / 'clean_dedup_fmri_to_t1_master.csv'
MANIFEST_CSV = PROJECT_ROOT / 'results' / 'dataset_creation' / 'bids_conversion_manifest.csv'
COMBINED_CSV = PROJECT_ROOT / 'results' / 'dataset_creation' / 'final_bids_dataset_manifest.csv'

master = pd.read_csv(MASTER_CSV)
manifest = pd.read_csv(MANIFEST_CSV)

print(f'Master shape:   {master.shape}')
print(f'Manifest shape: {manifest.shape}')


Master shape:   (3278, 33)
Manifest shape: (3278, 12)


## 1. Remove Duplicate `_T1wa` Files & Verify Only Standard BIDS Files Exist

In [2]:
# Check for any *_T1wa.* or other non-standard files inside session directories
removed_extra_files = []
for sub_dir in sorted(BIDS_ROOT.glob('sub-*')):
    if not sub_dir.is_dir():
        continue
    sub = sub_dir.name
    for ses_dir in sorted(sub_dir.glob('ses-*')):
        if not ses_dir.is_dir():
            continue
        ses = ses_dir.name
        allowed = {
            f'{sub}_{ses}_task-rest_bold.nii.gz',
            f'{sub}_{ses}_task-rest_bold.json',
            f'{sub}_{ses}_T1w.nii.gz',
            f'{sub}_{ses}_T1w.json',
        }
        for f in sorted(ses_dir.rglob('*')):
            if f.is_file() and f.name not in allowed:
                f.unlink()
                removed_extra_files.append(str(f.relative_to(PROJECT_ROOT)))

print(f'Removed {len(removed_extra_files)} non-standard duplicate BIDS file(s):')
for p in removed_extra_files:
    print(f'  - {p}')


Removed 2 non-standard duplicate BIDS file(s):
  - data\raw_bids\sub-006S4485\ses-20180925\anat\sub-006S4485_ses-20180925_T1wa.json
  - data\raw_bids\sub-006S4485\ses-20180925\anat\sub-006S4485_ses-20180925_T1wa.nii.gz


## 2. Remove the 11 Failed fMRI Session Folders (`anat`-Only Sessions) from `data/raw_bids`

In [3]:
failed_rows = manifest[(manifest['fMRI_Converted'] != True) | (manifest['T1_Converted'] != True)]
print(f'Failed fMRI rows in manifest: {len(failed_rows)}')

removed_sessions = []
for _, row in failed_rows.iterrows():
    sub = str(row['BIDS_Subject'])
    ses = str(row['BIDS_Session'])
    ses_dir = BIDS_ROOT / sub / ses
    if ses_dir.exists():
        shutil.rmtree(ses_dir)
        removed_sessions.append(f'{sub}/{ses} ({row["fMRI_Image_ID"]})')
    # Check if subject still has other valid sessions
    sub_dir = BIDS_ROOT / sub
    if sub_dir.exists() and not any(sub_dir.iterdir()):
        sub_dir.rmdir()

print(f'Removed {len(removed_sessions)} T1w-only failed session folder(s):')
for s in removed_sessions:
    print(f'  - {s}')

# Verify remaining session folders in data/raw_bids
all_ses_dirs = [p for p in BIDS_ROOT.glob('sub-*/ses-*') if p.is_dir()]
complete_pairs = 0
incomplete_dirs = []
for ses_dir in all_ses_dirs:
    sub = ses_dir.parent.name
    ses = ses_dir.name
    req = [
        ses_dir / 'func' / f'{sub}_{ses}_task-rest_bold.nii.gz',
        ses_dir / 'func' / f'{sub}_{ses}_task-rest_bold.json',
        ses_dir / 'anat' / f'{sub}_{ses}_T1w.nii.gz',
        ses_dir / 'anat' / f'{sub}_{ses}_T1w.json',
    ]
    if all(p.exists() for p in req):
        complete_pairs += 1
    else:
        incomplete_dirs.append(str(ses_dir))

print(f'\nTotal session folders remaining in data/raw_bids: {len(all_ses_dirs)}')
print(f'Complete (fMRI + T1w + JSONs) session folders:    {complete_pairs}')
print(f'Incomplete session folders:                       {len(incomplete_dirs)}')


Failed fMRI rows in manifest: 11
Removed 11 T1w-only failed session folder(s):
  - sub-002S1261/ses-20140313 (I418009)
  - sub-006S0731/ses-20181010 (I1058017)
  - sub-006S6252/ses-20190506 (I1162267)
  - sub-006S6500/ses-20180730 (I1028502)
  - sub-006S6681/ses-20190212 (I1148835)
  - sub-031S0618/ses-20181001 (I1055464)
  - sub-053S4578/ses-20160321 (I659915)
  - sub-053S4661/ses-20140617 (I431143)
  - sub-123S4127/ses-20211010 (I1504396)
  - sub-130S2391/ses-20111216 (I272899)
  - sub-136S4517/ses-20120914 (I335381)

Total session folders remaining in data/raw_bids: 3267
Complete (fMRI + T1w + JSONs) session folders:    3267
Incomplete session folders:                       0


## 3. Combine `clean_dedup_fmri_to_t1_master.csv` and `bids_conversion_manifest.csv` into One Unified CSV

In [4]:
# Verify 1-to-1 row alignment before merging
assert (master['Image Data ID'] == manifest['fMRI_Image_ID']).all(), 'fMRI ID mismatch!'
assert (master['Subject'] == manifest['Subject']).all(), 'Subject mismatch!'
assert (master['T1_Image_ID'] == manifest['T1_Image_ID']).all(), 'T1 ID mismatch!'
assert (master['Acq Date'] == manifest['fMRI_Acq_Date']).all(), 'fMRI Date mismatch!'
assert (master['T1_Study_Date'] == manifest['T1_Study_Date']).all(), 'T1 Date mismatch!'

# 1. Sync Diagnosis into bids_conversion_manifest.csv
manifest['Diagnosis'] = master['Diagnosis']
manifest.to_csv(MANIFEST_CSV, index=False)
print(f'Updated {MANIFEST_CSV.name} with Diagnosis column ({manifest.shape[0]} rows, {manifest.shape[1]} cols).')

# 2. Build unified table containing all information from both files
bids_only_cols = [
    'BIDS_Subject',
    'BIDS_Session',
    'fMRI_Converted',
    'fMRI_Status',
    'T1_Converted',
    'T1_Status',
    'Comments',
]
combined = pd.concat([master, manifest[bids_only_cols]], axis=1)
combined.to_csv(COMBINED_CSV, index=False)

print(f'Saved unified manifest to: {COMBINED_CSV}')
print(f'Combined shape: {combined.shape[0]} rows x {combined.shape[1]} columns')
print('\nColumns in final_bids_dataset_manifest.csv:')
for i, col in enumerate(combined.columns, 1):
    print(f'  {i:2d}. {col}')


Updated bids_conversion_manifest.csv with Diagnosis column (3278 rows, 13 cols).
Saved unified manifest to: D:\Mahdi\OU\Alzheimer's Disease\gnn-connectivity-matrix\results\dataset_creation\final_bids_dataset_manifest.csv
Combined shape: 3278 rows x 40 columns

Columns in final_bids_dataset_manifest.csv:
   1. Image Data ID
   2. Subject
   3. Group
   4. Sex
   5. Age
   6. Visit
   7. Modality
   8. Description
   9. Acq Date
  10. raw_id
  11. LONIImage
  12. VISCODE2
  13. StudyDate
  14. SeriesTime
  15. SeriesNumber
  16. SeriesType
  17. MRIProtocolPhase
  18. ScannerManufacturer
  19. ScannerModel
  20. MagneticFieldStrength
  21. Acceleration
  22. NumberVolumes
  23. SlicesPerVolume
  24. SliceThickness
  25. LONIStudy
  26. ReceiveCoilName
  27. T1_Image_ID
  28. T1_Raw_ID
  29. T1_Study_Date
  30. T1_Description
  31. Same_LONIStudy
  32. Days_Apart
  33. Diagnosis
  34. BIDS_Subject
  35. BIDS_Session
  36. fMRI_Converted
  37. fMRI_Status
  38. T1_Converted
  39. T1_Status

## 4. Create `participants.tsv` and `participants.json` in `data/raw_bids`

In [5]:
# Create BIDS-compliant participants.tsv for all subjects present in data/raw_bids
valid_combined = combined[(combined['fMRI_Converted'] == True) & (combined['T1_Converted'] == True)].copy()
valid_combined['acq_dt'] = pd.to_datetime(valid_combined['Acq Date'])
valid_combined = valid_combined.sort_values(['BIDS_Subject', 'acq_dt'])

participants_df = (
    valid_combined.groupby('BIDS_Subject', as_index=False)
    .first()[['BIDS_Subject', 'Subject', 'Sex', 'Age', 'Group', 'Diagnosis']]
    .rename(columns={
        'BIDS_Subject': 'participant_id',
        'Subject': 'adni_ptid',
        'Sex': 'sex',
        'Age': 'age_at_first_scan',
        'Group': 'enrollment_group',
        'Diagnosis': 'diagnosis_at_first_scan',
    })
)

participants_tsv = BIDS_ROOT / 'participants.tsv'
participants_df.to_csv(participants_tsv, sep='\t', index=False)

participants_json = BIDS_ROOT / 'participants.json'
participants_meta = {
    'participant_id': {'Description': 'BIDS participant identifier'},
    'adni_ptid': {'Description': 'Original ADNI Participant ID (PTID)'},
    'sex': {'Description': 'Biological sex of participant', 'Levels': {'M': 'Male', 'F': 'Female'}},
    'age_at_first_scan': {'Description': 'Participant age in years at their first fMRI scan in the dataset', 'Units': 'years'},
    'enrollment_group': {'Description': 'Initial ADNI enrollment cohort from LONI IDA metadata'},
    'diagnosis_at_first_scan': {
        'Description': 'Clinical consensus diagnosis (DXSUM) at the first fMRI scan session',
        'Levels': {'CN': 'Cognitively Normal', 'MCI': 'Mild Cognitive Impairment', 'AD': 'Alzheimer Disease / Dementia'}
    }
}
participants_json.write_text(json.dumps(participants_meta, indent=2))

print(f'Created {participants_tsv.name} ({len(participants_df)} unique subjects) and {participants_json.name} in {BIDS_ROOT}')
display(combined[['Subject', 'BIDS_Subject', 'BIDS_Session', 'Image Data ID', 'T1_Image_ID', 'Diagnosis', 'fMRI_Converted', 'T1_Converted', 'Comments']].head(10))


Created participants.tsv (1329 unique subjects) and participants.json in D:\Mahdi\OU\Alzheimer's Disease\gnn-connectivity-matrix\data\raw_bids
      Subject  BIDS_Subject  BIDS_Session Image Data ID T1_Image_ID Diagnosis  fMRI_Converted  T1_Converted                                                                                                                                                                                     Comments
0  002_S_0295  sub-002S0295  ses-20110602       I238623     I238627        CN            True          True                                                                                                                                                         converted with no problems first try
1  002_S_0295  sub-002S0295  ses-20120510       I303069     I303066        CN            True          True                                                                                                                                                         con